# Categorizer Exploration — With Canadian Merchant List
Same as the filtered notebook, but with a supplementary CSV of ~200 well-known Canadian merchants merged into the training data before training. This targets the vocabulary-coverage gap (Dollarama, A&W, etc.) that the base dataset doesn't cover — but it's a GENERIC list, not your own transaction history, so it likely won't fix hyper-local merchants (Pizzomatic, Jugo Juice, Kinjo, etc.). That's still a separate step.

In [ ]:
DATASET_NAME = "DoDataThings/us-bank-transaction-categories-v2"  # MIT licensed, not gated
CANADIAN_MERCHANTS_CSV = "canadian_merchants.csv"
REAL_CSV_PATHS = [
    "transactions.csv",
    "../../canadian_merchants.csv",
]

In [2]:
# pip install datasets   (new dependency, not previously needed)
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

c:\VSC_FIles\FinSight\backend\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load dataset and inspect actual category values
No manual download needed — this dataset isn't gated. IMPORTANT: confirm these printed category names match what's assumed in the mapping below before trusting it — I could not verify the exact strings myself (no HF network access in my sandbox).

In [3]:
ds = load_dataset(DATASET_NAME)
df = ds["train"].to_pandas()
print(df.shape)
print(df.columns.tolist())
df.head()

(68000, 2)
['description', 'category']


,description,category
0,[debit] NORTHWESTERN MUTUALEFT PYMT PPD ID: 99...,Insurance
1,[debit] MTG PMT PENFED CU,Mortgage
2,[debit] BETMGM 147 PARK BLVD UNION CITY 94587 ...,Entertainment
3,[debit] REPUBLIC SERVICES 222 MISSION CT CHICA...,Utilities
4,[debit] PYPL*365 MARKET,Groceries


In [4]:
df["category"].value_counts()

category
Insurance         4000
Rent              4000
Transfer          4000
Restaurants       4000
Education         4000
Transportation    4000
Income            4000
Fees              4000
Healthcare        4000
Mortgage          4000
Subscription      4000
Shopping          4000
Personal Care     4000
Groceries         4000
Utilities         4000
Entertainment     4000
Travel            4000
Name: count, dtype: int64

## 2. Apply category map, stratified sample
Mapping per your decisions: Rent/Mortgage and Education are new, untested categories. Transfer is intentionally excluded — that's not a bug, it's handled by `transfer_detector.py`'s heuristic and should never reach the ML model.

In [5]:
CATEGORY_MAP = {
    "Restaurants": "Food",
    "Groceries": "Groceries",
    "Transportation": "Transport",
    "Shopping": "Shopping",
    "Personal Care": "Shopping",
    "Entertainment": "Entertainment",
    "Utilities": "Utilities",
    "Subscription": "Utilities",
    "Insurance": "Utilities",
    "Healthcare": "Health",
    "Income": "Income",
    "Mortgage": "Rent/Mortgage",
    "Rent": "Rent/Mortgage",
    "Education": "Education",
    "Travel": "Other",
    "Fees": "Other",
    # "Transfer" intentionally omitted — heuristic-only, never ML-predicted
}

df["mapped_category"] = df["category"].map(CATEGORY_MAP)
unmapped = df[df["mapped_category"].isna()]["category"].unique()
print(f"{df['mapped_category'].isna().sum()} rows failed to map. Unmapped source categories: {list(unmapped)}")
print("(Transfer showing up here is expected — everything else should not be.)")
df["mapped_category"].value_counts()

4000 rows failed to map. Unmapped source categories: ['Transfer']
(Transfer showing up here is expected — everything else should not be.)


mapped_category
Utilities        12000
Rent/Mortgage     8000
Shopping          8000
Other             8000
Entertainment     4000
Groceries         4000
Health            4000
Income            4000
Transport         4000
Education         4000
Food              4000
Name: count, dtype: int64

In [6]:
SAMPLE_PER_CATEGORY = 12500

sampled = (
    df.dropna(subset=["mapped_category"])
    .groupby("mapped_category", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_CATEGORY), random_state=42))
)
sampled["mapped_category"].value_counts()

C:\Users\pkunj\AppData\Local\Temp\ipykernel_24096\4091788520.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_CATEGORY), random_state=42))


mapped_category
Utilities        12000
Other             8000
Rent/Mortgage     8000
Shopping          8000
Education         4000
Entertainment     4000
Food              4000
Groceries         4000
Health            4000
Income            4000
Transport         4000
Name: count, dtype: int64

## 2b. Merge in the Canadian merchant list
Each row repeated `REPEAT_COUNT` times before merging, so a single merchant label isn't outweighed by thousands of unrelated training rows. Testing whether repetition alone fixes cases like A&W/GYMVMT that were in the list but still mispredicted.

In [7]:
REPEAT_COUNT = 8  # each merchant row appears this many times in training

merchants_df = pd.read_csv(CANADIAN_MERCHANTS_CSV)
merchants_df = merchants_df.rename(columns={"category": "mapped_category"})
merchants_df = pd.concat([merchants_df] * REPEAT_COUNT, ignore_index=True)

print(f"Adding {len(merchants_df)} merchant rows ({len(merchants_df) // REPEAT_COUNT} unique x {REPEAT_COUNT}) to {len(sampled)} sampled training rows")
sampled = pd.concat([sampled[["description", "mapped_category"]], merchants_df], ignore_index=True)
sampled["mapped_category"].value_counts()

Adding 1664 merchant rows (208 unique x 8) to 64000 sampled training rows


mapped_category
Utilities        12288
Shopping          8368
Other             8088
Rent/Mortgage     8040
Food              4464
Transport         4120
Entertainment     4112
Health            4096
Education         4088
Groceries         4000
Income            4000
Name: count, dtype: int64

## 3. Train TF-IDF + LogisticRegression, check classification_report

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    sampled["description"], sampled["mapped_category"],
    test_size=0.2, random_state=42, stratify=sampled["mapped_category"]
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000)),
])

pipeline.fit(X_train, y_train)
print(classification_report(y_test, pipeline.predict(X_test)))

               precision    recall  f1-score   support

    Education       1.00      1.00      1.00       817
Entertainment       1.00      1.00      1.00       822
         Food       0.99      0.99      0.99       893
    Groceries       0.98      0.97      0.98       800
       Health       0.99      0.99      0.99       819
       Income       1.00      1.00      1.00       800
        Other       1.00      1.00      1.00      1618
Rent/Mortgage       1.00      1.00      1.00      1608
     Shopping       0.99      0.99      0.99      1674
    Transport       1.00      0.99      0.99       824
    Utilities       0.99      1.00      0.99      2458

     accuracy                           0.99     13133
    macro avg       0.99      0.99      0.99     13133
 weighted avg       0.99      0.99      0.99     13133



## 4. Filter real transactions through the transfer/savings heuristic
Copied from `app/ml/transfer_detector.py` — keep these two in sync if that file changes. Rows caught here never reach the ML model in production, so they're excluded before we look at prediction quality.

In [9]:
SAVINGS_KEYWORDS = ["TO FIND & SAVE"]
TRANSFER_KEYWORDS = [
    "E-TRANSFER",
    "ONLINE BANKING TRANSFER",
    "BILL PAYMENT",
    "ONLINE TRANSFER",
    "PAYMENT - THANK YOU",
]

def is_savings(description):
    description = description.upper()
    return any(k in description for k in SAVINGS_KEYWORDS)

def is_transfer(description):
    description = description.upper()
    return any(k in description for k in TRANSFER_KEYWORDS)

In [10]:
real_frames = []
for path in REAL_CSV_PATHS:
    r = pd.read_csv(path)
    r["description"] = (r["Description 1"].fillna("") + " " + r["Description 2"].fillna("")).str.strip()
    r["source_file"] = path
    real_frames.append(r)

real = pd.concat(real_frames, ignore_index=True)

real["is_savings"] = real["description"].apply(is_savings)
real["is_transfer"] = real["description"].apply(is_transfer)

total = len(real)
savings_count = real["is_savings"].sum()
transfer_count = (real["is_transfer"] & ~real["is_savings"]).sum()  # savings checked first, same order as production
ml_remaining = total - savings_count - transfer_count

print(f"Total rows: {total}")
print(f"Caught by savings heuristic: {savings_count} ({savings_count/total*100:.1f}%)")
print(f"Caught by transfer heuristic: {transfer_count} ({transfer_count/total*100:.1f}%)")
print(f"Remaining for ML: {ml_remaining} ({ml_remaining/total*100:.1f}%)")

Total rows: 1016
Caught by savings heuristic: 153 (15.1%)
Caught by transfer heuristic: 528 (52.0%)
Remaining for ML: 335 (33.0%)


## 5. Predict only on what actually reaches the ML model

In [11]:
ml_rows = real[~real["is_savings"] & ~real["is_transfer"]].copy()
ml_rows["predicted_category"] = pipeline.predict(ml_rows["description"])
print_csv=ml_rows[["source_file", "description", "predicted_category"]]
print_csv

,source_file,description,predicted_category
11,transactions.csv,MISC PAYMENT RBC OFFER/OFFRE,Rent/Mortgage
14,transactions.csv,ATM DEPOSIT - EC291652,Other
17,transactions.csv,ONLINE BANKING PAYMENT - 3302 SAIT - TUITION,Education
64,transactions.csv,BUSINESS PAD NORTHERN ALBERT,Food
80,transactions.csv,MISC PAYMENT PCF LINK DEP,Income
...,...,...,...
1010,download-transactions.csv,TRANSIT-ESTORE CALGARY,Transport
1011,download-transactions.csv,GRANNYS CHEESECAKE 1 CALGARY,Food
1012,download-transactions.csv,NORTH AMERICAN MIDWAY BRANTFORD,Utilities
1014,download-transactions.csv,SQ *TACO NORI Calgary,Food


In [12]:
print_csv.to_csv("predicted_categories.csv", index=False)